# UK Biobank Neurodegeneration Literature Review (Clean Notebook)

This notebook is a cleaned, reproducible version of the exploratory workflow.
It keeps the full pipeline in one place for paper review.


## Workflow
1. (Optional) Scrape UK Biobank publications by year
2. (Optional) Enrich records with DOI/abstract from Crossref
3. Filter records with neurodegeneration keywords
4. Clean text and build analysis-ready fields
5. Cluster topics and generate review figures


In [ ]:
from pathlib import Path
import re
import time
import warnings

import numpy as np
import pandas as pd
import requests
from bs4 import BeautifulSoup
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid")


In [ ]:
PROJECT_ROOT = Path(".").resolve()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
FIG_DIR = PROJECT_ROOT / "outputs" / "figures"

for folder in [RAW_DIR, INTERIM_DIR, PROCESSED_DIR, FIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

INPUT_WITH_ABSTRACTS = PROJECT_ROOT / "Parkinsons" / "UKBB_All_Publications_With_DOI_Abstracts.csv"
FILTERED_OUT = INTERIM_DIR / "dementia_related_papers.csv"
CLEANED_OUT = PROCESSED_DIR / "dementia_related_papers_cleaned.csv"
CLUSTERED_OUT = PROCESSED_DIR / "dementia_related_papers_clustered.csv"

RUN_SCRAPE = False
RUN_CROSSREF = False
CROSSREF_EMAIL = "your_email@example.com"  # Update if RUN_CROSSREF=True

START_YEAR = 2013
END_YEAR = 2025
N_CLUSTERS = 12
MAX_FEATURES = 1000
RANDOM_STATE = 42


## Helpers: schema normalization


In [ ]:
MISSING_SENTINELS = {"", "not found", "doi not found", "abstract not found", "nan", "none"}


def normalize_name(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(name).strip().lower()).strip("_")


def first_column(df: pd.DataFrame, *candidates: str) -> pd.Series:
    norm_map = {}
    for col in df.columns:
        key = normalize_name(col)
        if key not in norm_map:
            norm_map[key] = col

    for candidate in candidates:
        key = normalize_name(candidate)
        if key in norm_map:
            return df[norm_map[key]]

    return pd.Series([pd.NA] * len(df), index=df.index)


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    out["publication_id"] = first_column(df, "publication_id", "pub_id", "pub id", "id")
    out["title"] = first_column(df, "title")
    out["authors"] = first_column(df, "authors", "author_s", "author(s)", "author")
    out["year"] = first_column(df, "year")
    out["source_year"] = first_column(df, "source_year", "year_1", "year.1")
    out["journal"] = first_column(df, "journal")
    out["doi"] = first_column(df, "doi")
    out["abstract"] = first_column(df, "abstract")

    optional_pairs = [
        ("cleaned_abstract", ("cleaned_abstract",)),
        ("text", ("text",)),
        ("cluster", ("cluster",)),
        ("cluster_keywords", ("cluster_keywords", "keywords")),
        ("matched_keywords", ("matched_keywords",)),
    ]
    for out_col, aliases in optional_pairs:
        series = first_column(df, *aliases)
        if not series.isna().all():
            out[out_col] = series

    if out["year"].isna().all() and not out["source_year"].isna().all():
        out["year"] = out["source_year"]
    if out["source_year"].isna().all() and not out["year"].isna().all():
        out["source_year"] = out["year"]

    for year_col in ["year", "source_year"]:
        out[year_col] = pd.to_numeric(out[year_col], errors="coerce").astype("Int64")

    text_columns = [
        "publication_id", "title", "authors", "journal", "doi",
        "abstract", "cleaned_abstract", "text", "cluster_keywords", "matched_keywords",
    ]
    for col in text_columns:
        if col in out.columns:
            out[col] = out[col].astype("string")

    if "cluster" in out.columns:
        out["cluster"] = pd.to_numeric(out["cluster"], errors="coerce").astype("Int64")

    return out


def is_missing(value) -> bool:
    if pd.isna(value):
        return True
    return str(value).strip().lower() in MISSING_SENTINELS


## Optional web steps
Set `RUN_SCRAPE=True` and/or `RUN_CROSSREF=True` only when needed.


In [ ]:
BASE_UKBB_URL = "https://biobank.ndph.ox.ac.uk/showcase/docs.cgi?id=2&year={year}"
CROSSREF_WORKS_URL = "https://api.crossref.org/works"


def scrape_ukbb_publications(start_year=2013, end_year=2025, sleep_seconds=0.2) -> pd.DataFrame:
    session = requests.Session()
    records = []

    for year in range(start_year, end_year + 1):
        response = session.get(BASE_UKBB_URL.format(year=year), timeout=30)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")
        table = soup.find("table")
        if table is None:
            continue

        headers = [normalize_name(th.get_text(" ", strip=True)) for th in table.find_all("th")]
        for row in table.find_all("tr")[1:]:
            cells = [td.get_text(" ", strip=True) for td in row.find_all("td")]
            if not cells:
                continue
            row_dict = dict(zip(headers, cells))
            records.append(
                {
                    "publication_id": row_dict.get("pub_id") or row_dict.get("id"),
                    "title": row_dict.get("title"),
                    "authors": row_dict.get("author_s") or row_dict.get("authors"),
                    "year": pd.to_numeric(row_dict.get("year"), errors="coerce"),
                    "source_year": year,
                    "journal": row_dict.get("journal"),
                }
            )
        if sleep_seconds:
            time.sleep(sleep_seconds)

    return standardize_columns(pd.DataFrame.from_records(records))


def crossref_lookup(title: str, journal: str = "", email: str = "", retries: int = 3, delay: float = 3.0):
    session = requests.Session()
    if email:
        session.headers.update({"User-Agent": f"UKBB-litreview-notebook/1.0 (mailto:{email})"})

    query_plan = [{"query.title": title, "rows": 1}]
    if journal:
        query_plan.insert(0, {"query.title": title, "query.container-title": journal, "rows": 1})

    for params in query_plan:
        for attempt in range(1, retries + 1):
            try:
                response = session.get(CROSSREF_WORKS_URL, params=params, timeout=30)
                response.raise_for_status()
                items = response.json().get("message", {}).get("items", [])
                if items:
                    first = items[0]
                    return first.get("DOI", ""), first.get("abstract", "")
                break
            except requests.RequestException:
                if attempt == retries:
                    break
                time.sleep(delay * attempt)

    return "DOI not found", "Abstract not found"


def enrich_with_crossref(df: pd.DataFrame, email: str, sleep_seconds: float = 0.2) -> pd.DataFrame:
    enriched = df.copy()
    if "doi" not in enriched.columns:
        enriched["doi"] = pd.NA
    if "abstract" not in enriched.columns:
        enriched["abstract"] = pd.NA

    for idx, row in enriched.iterrows():
        if not is_missing(row.get("doi")) and not is_missing(row.get("abstract")):
            continue

        title = str(row.get("title", "") or "").strip()
        journal = str(row.get("journal", "") or "").strip()
        if not title:
            continue

        doi, abstract = crossref_lookup(title=title, journal=journal, email=email)
        enriched.at[idx, "doi"] = doi or "DOI not found"
        enriched.at[idx, "abstract"] = abstract or "Abstract not found"

        if sleep_seconds:
            time.sleep(sleep_seconds)

    return standardize_columns(enriched)


if RUN_SCRAPE:
    scraped_df = scrape_ukbb_publications(START_YEAR, END_YEAR)
    scraped_df.to_csv(RAW_DIR / "ukbb_publications_by_year.csv", index=False)

if RUN_CROSSREF:
    if CROSSREF_EMAIL == "your_email@example.com":
        raise ValueError("Set CROSSREF_EMAIL before RUN_CROSSREF=True")
    source_for_crossref = scraped_df if RUN_SCRAPE else standardize_columns(pd.read_csv(INPUT_WITH_ABSTRACTS))
    enriched_df = enrich_with_crossref(source_for_crossref, email=CROSSREF_EMAIL)
    enriched_df.to_csv(RAW_DIR / "ukbb_all_publications_with_doi_abstracts.csv", index=False)


## Core analysis: filter, clean, cluster


In [ ]:
KEYWORDS = [
    "dementia", "alzheimer", "parkinson", "huntington", "mci",
    "mild cognitive impairment", "frontotemporal dementia",
    "tbi", "traumatic brain injury", "lewy body dementia",
    "vascular dementia", "neurodegenerative",
]

EXCLUDED_TERMS = {
    "abstract", "conclusion", "introduction", "results", "methods",
    "discussion", "background", "objective", "ci",
}

ABBREVIATIONS = {
    "pd": "parkinson",
    "ad": "alzheimer",
    "mci": "mild cognitive impairment",
    "tbi": "traumatic brain injury",
    "ftd": "frontotemporal dementia",
    "snca": "synuclein alpha",
}


def filter_by_keywords(df: pd.DataFrame, keywords=KEYWORDS) -> pd.DataFrame:
    keywords = [k.lower() for k in keywords]

    def matches(text) -> list[str]:
        if is_missing(text):
            return []
        text_lower = str(text).lower()
        return [k for k in keywords if k in text_lower]

    title_matches = df["title"].apply(matches)
    abstract_matches = df["abstract"].apply(matches)
    journal_matches = df["journal"].apply(matches)

    all_matches = [
        sorted(set(t + a + j))
        for t, a, j in zip(title_matches, abstract_matches, journal_matches)
    ]
    mask = pd.Series([bool(items) for items in all_matches], index=df.index)

    filtered = df.loc[mask].copy()
    filtered["matched_keywords"] = [", ".join(items) for items in np.array(all_matches, dtype=object)[mask.values]]
    return standardize_columns(filtered)


def clean_abstract(raw_text) -> str:
    if is_missing(raw_text):
        return ""

    plain = BeautifulSoup(str(raw_text), "html.parser").get_text(separator=" ")
    words = re.findall(r"\w+", plain.lower())

    cleaned_words = []
    for word in words:
        if word in EXCLUDED_TERMS or word in ENGLISH_STOP_WORDS or word.isnumeric():
            continue
        replacement = ABBREVIATIONS.get(word, word)
        cleaned_words.extend(replacement.split())

    return " ".join(cleaned_words)


def prepare_text(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["cleaned_abstract"] = out["abstract"].apply(clean_abstract)
    out["text"] = (
        out["title"].fillna("").astype(str).str.strip()
        + " "
        + out["cleaned_abstract"].fillna("").astype(str).str.strip()
    ).str.strip()
    return standardize_columns(out)


def top_keywords_per_cluster(labels: pd.Series, tfidf_matrix, vectorizer: TfidfVectorizer, n_keywords=5):
    feature_names = vectorizer.get_feature_names_out()
    output = {}

    for cluster_id in sorted(labels.dropna().astype(int).unique().tolist()):
        mask = (labels.astype("Int64") == cluster_id).to_numpy()
        cluster_matrix = tfidf_matrix[mask, :]
        mean_scores = np.asarray(cluster_matrix.mean(axis=0)).ravel()
        top_idx = np.argsort(mean_scores)[-n_keywords:][::-1]
        output[cluster_id] = [feature_names[i] for i in top_idx]

    return output


def build_cluster_color_map(cluster_series: pd.Series) -> dict[int, tuple[float, float, float]]:
    """Match the original figure colors exactly using tab20 indices 0..11."""
    cluster_ids = sorted(cluster_series.dropna().astype(int).unique().tolist())
    base_tab20 = sns.color_palette("tab20", n_colors=20)

    # Cluster 1..12 map to the first 12 tab20 colors in order:
    # blue, light-blue, orange, light-orange, green, light-green,
    # red, light-red, purple, light-purple, brown, light-brown.
    palette = base_tab20[:12]

    if len(cluster_ids) > len(palette):
        raise ValueError(
            f"Detected {len(cluster_ids)} clusters but fixed palette supports 12. "
            "Set N_CLUSTERS=12 for figure-consistent colors."
        )

    return {cluster_id: palette[i] for i, cluster_id in enumerate(cluster_ids)}


def cluster_topics(df: pd.DataFrame, n_clusters=12, max_features=1000, random_state=42, n_keywords=5):
    vectorizer = TfidfVectorizer(stop_words="english", max_features=max_features)
    tfidf = vectorizer.fit_transform(df["text"].fillna(""))

    model = KMeans(n_clusters=n_clusters, random_state=random_state, n_init=10)
    labels = model.fit_predict(tfidf) + 1

    out = df.copy()
    out["cluster"] = labels

    cluster_kw = top_keywords_per_cluster(out["cluster"], tfidf, vectorizer, n_keywords=n_keywords)
    out["cluster_keywords"] = out["cluster"].map(lambda c: ", ".join(cluster_kw[int(c)]))

    n_samples = tfidf.shape[0]
    perplexity = min(30, max(5, (n_samples - 1) // 3))
    tsne = TSNE(
        n_components=2,
        random_state=random_state,
        perplexity=perplexity,
        init="pca",
        learning_rate="auto",
    )
    embedding = tsne.fit_transform(tfidf.toarray())

    return standardize_columns(out), embedding, cluster_kw, vectorizer, tfidf


def plot_tsne(df: pd.DataFrame, embedding: np.ndarray, kw_map: dict, color_map: dict, out_path: Path):
    cluster_colors = [color_map[int(c)] for c in df["cluster"].astype(int)]

    plt.figure(figsize=(8, 8))
    plt.scatter(
        embedding[:, 0],
        embedding[:, 1],
        c=cluster_colors,
        alpha=0.9,
        s=35,
    )
    plt.title("t-SNE Visualization of Dementia-Related Clusters")
    plt.xlabel("Dimension 1")
    plt.ylabel("Dimension 2")
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.show()


def plot_stacked_by_year(df: pd.DataFrame, kw_map: dict, color_map: dict, out_path: Path):
    cluster_ids = sorted(df["cluster"].dropna().astype(int).unique().tolist())
    grouped = df.groupby(["year", "cluster"]).size().unstack(fill_value=0).sort_index()
    grouped = grouped.reindex(columns=cluster_ids, fill_value=0)

    plt.figure(figsize=(9, 6))
    bottoms = np.zeros(len(grouped.index))

    for cluster_id in cluster_ids:
        values = grouped[cluster_id].to_numpy()
        plt.bar(
            grouped.index.astype(int),
            values,
            bottom=bottoms,
            color=color_map[cluster_id],
            alpha=0.9,
            label=f"Cluster {cluster_id}: {', '.join(kw_map[cluster_id])}",
        )
        bottoms += values

    plt.title("Number of Publications per Year with Cluster Contribution")
    plt.xlabel("Year")
    plt.ylabel("Number of Publications")
    plt.xticks(grouped.index.astype(int), rotation=45)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.show()


def _cluster_tfidf_means(df: pd.DataFrame, tfidf_matrix, vectorizer: TfidfVectorizer):
    tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=vectorizer.get_feature_names_out(), index=df.index)
    cluster_ids = sorted(df["cluster"].dropna().astype(int).unique().tolist())

    cluster_weights = {}
    for cluster_id in cluster_ids:
        cluster_idx = df[df["cluster"].astype("Int64") == cluster_id].index
        mean_scores = tfidf_df.loc[cluster_idx].mean()
        cluster_weights[cluster_id] = {word: float(score) for word, score in mean_scores.items() if score > 0}

    return cluster_weights


def plot_wordcloud_grid(df: pd.DataFrame, tfidf_matrix, vectorizer: TfidfVectorizer, color_map: dict, out_path: Path):
    try:
        from wordcloud import WordCloud
    except ImportError:
        print("wordcloud is not installed. Run: pip install wordcloud")
        return

    cluster_ids = sorted(df["cluster"].dropna().astype(int).unique().tolist())
    weights_by_cluster = _cluster_tfidf_means(df, tfidf_matrix, vectorizer)

    cols = 3
    rows = int(np.ceil(len(cluster_ids) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(18, 5 * rows))
    axes = np.atleast_1d(axes).flatten()

    for i, cluster_id in enumerate(cluster_ids):
        rgb = tuple(int(255 * ch) for ch in color_map[cluster_id])

        def color_func(*args, _rgb=rgb, **kwargs):
            return f"rgb({_rgb[0]}, {_rgb[1]}, {_rgb[2]})"

        cloud = WordCloud(
            width=500,
            height=500,
            background_color="white",
            max_words=100,
            color_func=color_func,
        ).generate_from_frequencies(weights_by_cluster[cluster_id])

        ax = axes[i]
        ax.imshow(cloud, interpolation="bilinear")
        ax.set_title(f"Word Cloud for Cluster {cluster_id}", fontsize=14)
        ax.axis("off")

    for j in range(len(cluster_ids), len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.show()


def plot_full_figure(df: pd.DataFrame, embedding: np.ndarray, kw_map: dict, tfidf_matrix, vectorizer: TfidfVectorizer, color_map: dict, out_path: Path):
    try:
        from wordcloud import WordCloud
    except ImportError:
        print("wordcloud is not installed. Run: pip install wordcloud")
        return

    from matplotlib.lines import Line2D

    cluster_ids = sorted(df["cluster"].dropna().astype(int).unique().tolist())
    weights_by_cluster = _cluster_tfidf_means(df, tfidf_matrix, vectorizer)

    fig = plt.figure(figsize=(24, 16))
    outer = fig.add_gridspec(1, 2, width_ratios=[2.2, 1.2], wspace=0.1)

    left = outer[0, 0].subgridspec(4, 3, wspace=0.02, hspace=0.08)
    wc_axes = []
    for i, cluster_id in enumerate(cluster_ids):
        ax = fig.add_subplot(left[i // 3, i % 3])
        wc_axes.append(ax)

        rgb = tuple(int(255 * ch) for ch in color_map[cluster_id])

        def color_func(*args, _rgb=rgb, **kwargs):
            return f"rgb({_rgb[0]}, {_rgb[1]}, {_rgb[2]})"

        cloud = WordCloud(
            width=500,
            height=500,
            background_color="white",
            max_words=100,
            color_func=color_func,
        ).generate_from_frequencies(weights_by_cluster[cluster_id])

        ax.imshow(cloud, interpolation="bilinear")
        ax.set_title(f"Word Cloud for Cluster {cluster_id}", fontsize=12)
        ax.axis("off")

    if wc_axes:
        wc_axes[0].text(-0.12, 1.12, "A", transform=wc_axes[0].transAxes, fontsize=24, fontweight="bold")

    right = outer[0, 1].subgridspec(3, 1, height_ratios=[1.0, 1.0, 0.75], hspace=0.3)

    ax_bar = fig.add_subplot(right[0, 0])
    grouped = df.groupby(["year", "cluster"]).size().unstack(fill_value=0).sort_index()
    grouped = grouped.reindex(columns=cluster_ids, fill_value=0)

    bottoms = np.zeros(len(grouped.index))
    for cluster_id in cluster_ids:
        values = grouped[cluster_id].to_numpy()
        ax_bar.bar(
            grouped.index.astype(int),
            values,
            bottom=bottoms,
            color=color_map[cluster_id],
            alpha=0.9,
        )
        bottoms += values

    ax_bar.set_title("Number of Publications per Year with Cluster Contribution")
    ax_bar.set_xlabel("Year")
    ax_bar.set_ylabel("Number of Publications")
    ax_bar.tick_params(axis="x", rotation=45)
    ax_bar.text(-0.16, 1.08, "B", transform=ax_bar.transAxes, fontsize=24, fontweight="bold")

    ax_tsne = fig.add_subplot(right[1, 0])
    cluster_colors = [color_map[int(c)] for c in df["cluster"].astype(int)]
    ax_tsne.scatter(embedding[:, 0], embedding[:, 1], c=cluster_colors, s=35, alpha=0.9)
    ax_tsne.set_title("t-SNE Visualization of Dementia-Related Clusters")
    ax_tsne.set_xlabel("Dimension 1")
    ax_tsne.set_ylabel("Dimension 2")
    ax_tsne.text(-0.16, 1.08, "C", transform=ax_tsne.transAxes, fontsize=24, fontweight="bold")

    ax_legend = fig.add_subplot(right[2, 0])
    ax_legend.axis("off")
    handles = [
        Line2D([0], [0], marker="s", linestyle="", markerfacecolor=color_map[cid], markeredgecolor=color_map[cid], markersize=10)
        for cid in cluster_ids
    ]
    labels = [f"Cluster {cid}: {', '.join(kw_map[cid])}" for cid in cluster_ids]
    ax_legend.legend(handles, labels, title="Clusters and Keywords", loc="upper left", frameon=True, fontsize=10, title_fontsize=13)

    plt.tight_layout()
    plt.savefig(out_path, dpi=300)
    plt.show()


source_df = standardize_columns(pd.read_csv(INPUT_WITH_ABSTRACTS))
filtered_df = filter_by_keywords(source_df)
prepared_df = prepare_text(filtered_df)
clustered_df, embedding, cluster_keywords, vectorizer, tfidf_matrix = cluster_topics(
    prepared_df,
    n_clusters=N_CLUSTERS,
    max_features=MAX_FEATURES,
    random_state=RANDOM_STATE,
)
cluster_color_map = build_cluster_color_map(clustered_df["cluster"])

print(f"Input rows:           {len(source_df):,}")
print(f"Keyword-filtered:     {len(filtered_df):,}")
print(f"Prepared/clustered:   {len(clustered_df):,}")
print(f"Clusters:             {sorted(clustered_df['cluster'].dropna().astype(int).unique().tolist())}")

plot_wordcloud_grid(
    clustered_df,
    tfidf_matrix,
    vectorizer,
    cluster_color_map,
    FIG_DIR / "combined_wordclouds.png",
)
plot_stacked_by_year(
    clustered_df,
    cluster_keywords,
    cluster_color_map,
    FIG_DIR / "stacked_bar_chart.png",
)
plot_tsne(
    clustered_df,
    embedding,
    cluster_keywords,
    cluster_color_map,
    FIG_DIR / "tsne_visualization.png",
)
plot_full_figure(
    clustered_df,
    embedding,
    cluster_keywords,
    tfidf_matrix,
    vectorizer,
    cluster_color_map,
    FIG_DIR / "full_figure.png",
)


## Save outputs


In [ ]:
filtered_df.to_csv(FILTERED_OUT, index=False)
prepared_df.to_csv(CLEANED_OUT, index=False)
clustered_df.to_csv(CLUSTERED_OUT, index=False)

print(f"Saved filtered dataset to:  {FILTERED_OUT}")
print(f"Saved cleaned dataset to:   {CLEANED_OUT}")
print(f"Saved clustered dataset to: {CLUSTERED_OUT}")
print(f"Saved figures to:           {FIG_DIR}")

summary = pd.DataFrame(
    [
        {"stage": "input", "rows": len(source_df)},
        {"stage": "keyword_filtered", "rows": len(filtered_df)},
        {"stage": "prepared", "rows": len(prepared_df)},
        {"stage": "clustered", "rows": len(clustered_df)},
    ]
)
summary
